# Offline Figure & Metric Regeneration Notebook

This notebook regenerates all **11 figures** and prints a disk-verified summary of headline metrics **100% offline** from committed JSON/JSONL files (zero GPU, zero `OPENAI_API_KEY`, and zero raw activation caches required).

**Data provenance:** `Qwen/Qwen3-4B` neuron explanations and causal recovery scores (`data/qwen4b/ulto85/*.json`) come from [Ulto85/neurons-kinda-clean](https://github.com/Ulto85/neurons-kinda-clean) (MIT License, verbatim at `data/qwen4b/ulto85/LICENSE`). Text snippets are sourced from `HuggingFaceFW/fineweb-edu` (`Qwen3-1.7B`) and `HuggingFaceFW/fineweb` (`Qwen3-4B`) under the **ODC-By** license.


In [1]:
import os
import sys
import warnings
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ.pop('OPENAI_API_KEY', None)
warnings.simplefilter('error', RuntimeWarning)

cwd = Path.cwd()
if (cwd / 'src' / 'w_audit').is_dir():
    REPO_ROOT = cwd
elif (cwd.parent / 'src' / 'w_audit').is_dir():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError('Run this notebook from the repository root or notebooks/ directory.')

src_dir = str(REPO_ROOT / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

npy_files = list(REPO_ROOT.rglob('*.npy'))
print(f'Offline mode confirmed | .npy caches present: {len(npy_files)} (0 required)')
print('Committed packages verified: w_audit, qwen4b_eval, qwen17b_causal')


Offline mode confirmed | .npy caches present: 0 (0 required)
Committed packages verified: w_audit, qwen4b_eval, qwen17b_causal


## Part 1 — `Qwen/Qwen3-1.7B` Multi-Band Detection Audit (`3` figures)

Regenerates `figures/main_band_accuracy.png`, `figures/w_band_accuracy.png`, and `figures/context_ablation.png` from committed `data/` and `results/` files.


In [2]:
from w_audit.analyze import main as analyze_17b_det_main

rc = analyze_17b_det_main(['--primary-judge', 'gpt-6.1-terra@none'])
assert rc == 0, f'w_audit.analyze returned {rc}'
print('Part 1 figures regenerated: figures/main_band_accuracy.png, figures/w_band_accuracy.png, figures/context_ablation.png')


Analysis complete (primary=gpt-6.1-terra@none): wrote figures\main_band_accuracy.png, figures\w_band_accuracy.png, results\summary_tables.md, and audit\raw_30_poles_worksheet.md. Pilot gate: GO (Layer-4 positive control >= 0.80 and cleanly beats wrong/no_label floors)
Part 1 figures regenerated: figures/main_band_accuracy.png, figures/w_band_accuracy.png, figures/context_ablation.png


## Part 2 — `Qwen/Qwen3-4B` (`Ulto85/neurons-kinda-clean`, MIT) Detection vs. Causal Audit (`4` figures)

Regenerates `figures/qwen4b_main_plain.png`, `figures/qwen4b_top100_two_panel.png`, `figures/qwen4b_detection_vs_causal.png`, and `figures/qwen4b_detection_vs_strong20.png`.


In [3]:
from qwen4b_eval.analyze_4b import main as analyze_4b_main

rc = analyze_4b_main([])
assert rc == 0, f'qwen4b_eval.analyze_4b returned {rc}'
print('Part 2 figures regenerated: qwen4b_main_plain.png, qwen4b_top100_two_panel.png, qwen4b_detection_vs_causal.png, qwen4b_detection_vs_strong20.png')


4B analysis complete: 278 neurons / 424 active poles, 36,198 calls ($15.4733); wrote results\qwen4b\summary_4b.md, results\qwen4b\metrics_4b.json, and figures.
Part 2 figures regenerated: qwen4b_main_plain.png, qwen4b_top100_two_panel.png, qwen4b_detection_vs_causal.png, qwen4b_detection_vs_strong20.png


## Exploratory Diagnostics — Activation-Energy Decomposition & Causal Surrogates (`2` figures)

Regenerates `figures/qwen4b_gap_decomposition.png` and `figures/qwen4b_surrogate_comparison.png`.


In [4]:
from qwen4b_eval.surrogate_4b import main as surrogate_4b_main

rc = surrogate_4b_main(['analyze', '--n-bootstrap', '1000'])
assert rc == 0, f'qwen4b_eval.surrogate_4b returned {rc}'
print('Exploratory figures regenerated: figures/qwen4b_gap_decomposition.png, figures/qwen4b_surrogate_comparison.png')


{
  "n_neurons": 278,
  "surrogate_spend": {
    "multiband_jsonl_calls": 18890,
    "multiband_spend_usd": 4.539567,
    "multiband_by_model_usd": {
      "gpt-6-luna@none": 0.616245,
      "gpt-6.1-terra@none": 3.923322
    },
    "fpr_jsonl_calls": 15264,
    "fpr_spend_usd": 2.064365,
    "fpr_by_model_usd": {
      "gpt-6-luna@none": 0.344031,
      "gpt-6.1-terra@none": 1.720334
    },
    "combined_spend_usd": 6.603932,
    "budget_cap_usd": 8.0,
    "within_budget": true
  },
  "best_zero_param_surrogate": "s4_wdet_top100_times_1m_fpr_terra",
  "best_overall_llm_surrogate": "s4_cv_rank_ridge_full_luna",
  "has_kl_validation": true
}
Exploratory figures regenerated: figures/qwen4b_gap_decomposition.png, figures/qwen4b_surrogate_comparison.png


## Cross-Model Exact `float32` Band KL & Inconclusive `1.7B` Causal Check (`1` figure)

Regenerates `figures/cross_model_17b_vs_4b.png`.


In [5]:
from qwen17b_causal.analyze_17b_causal import main as analyze_17b_causal_main

rc = analyze_17b_causal_main(['--n-bootstrap', '1000'])
assert rc == 0, f'qwen17b_causal.analyze_17b_causal returned {rc}'
print('Cross-model figure regenerated: figures/cross_model_17b_vs_4b.png')


{
  "n_neurons": 50,
  "causal_17b_spend": {
    "sim_calls": 3200,
    "sim_spend_usd": 3.306358,
    "sim_parse_status": {
      "ok": 3121,
      "empty_ok": 63,
      "fallback_regex": 16
    },
    "sim_reasoning_tokens": 0,
    "fpr_calls": 3600,
    "fpr_spend_usd": 0.332253,
    "fpr_reasoning_tokens": 0,
    "combined_spend_usd": 3.638611,
    "budget_cap_usd": 4.5,
    "within_budget": true
  },
  "causal_distribution_17b": {
    "median_sim_cal_rec": -0.029177848791978267,
    "mean_sim_cal_rec": -0.18006550461073517,
    "median_sim_cal_rec_cv": -0.029177848791978267,
    "median_sim_cal_rec_insample": -0.019946448100619962,
    "median_sim_cal_rec_active": 0.033612693258699144,
    "mean_sim_cal_rec_active": 0.060333598805724045,
    "median_strong20_rec": 0.3555985525247947,
    "median_energy_ge_20pct_max": 0.27358081264141865
  }
}
Cross-model figure regenerated: figures/cross_model_17b_vs_4b.png


## Primary Cross-Model Synthesis — Effect-Weighted Detection Lift (`1` figure)

Regenerates `figures/main_detection_vs_kl_by_band.png` and `results/qwen4b/effect_weighted.json`.


In [6]:
from qwen4b_eval.effect_weighted import main as effect_weighted_main

rc = effect_weighted_main(['--skip-reading-sheet'])
assert rc == 0, f'qwen4b_eval.effect_weighted returned {rc}'
print('Primary synthesis figure regenerated: figures/main_detection_vs_kl_by_band.png')


effect_weighted complete (4B n=278, 1.7B n=50): 4B EWL=0.345 (2-sided=0.359, >=p75=0.421, p99.9+ lift=0.679); 1.7B EWL=0.397 (p99.9+ lift=0.563, raw over no_label=+0.205) -> wrote results\qwen4b\effect_weighted.json & figures\main_detection_vs_kl_by_band.png
Primary synthesis figure regenerated: figures/main_detection_vs_kl_by_band.png


## Final Offline Verification Table (All 11 Figures & Disk-Recomputed Headline Metrics)


In [7]:
import json
import struct
from pathlib import Path

EXPECTED_FIGURES = [
    'figures/main_detection_vs_kl_by_band.png',
    'figures/main_band_accuracy.png',
    'figures/w_band_accuracy.png',
    'figures/context_ablation.png',
    'figures/qwen4b_main_plain.png',
    'figures/qwen4b_top100_two_panel.png',
    'figures/qwen4b_detection_vs_causal.png',
    'figures/qwen4b_detection_vs_strong20.png',
    'figures/qwen4b_gap_decomposition.png',
    'figures/qwen4b_surrogate_comparison.png',
    'figures/cross_model_17b_vs_4b.png',
]

print(f'{"Figure":46s} {"Dims (px)":>12s} {"Size (KB)":>10s}')
print('-' * 72)
for rel in EXPECTED_FIGURES:
    p = Path(rel)
    assert p.is_file(), f'Missing regenerated figure: {rel}'
    raw = p.read_bytes()
    assert raw[:8] == b'\x89PNG\r\n\x1a\n', f'Invalid PNG header: {rel}'
    w, h = struct.unpack('>II', raw[16:24])
    print(f'{rel:46s} {w:5d}x{h:<6d} {len(raw)/1024:10.1f}')

m17 = json.loads(Path('results/metrics.json').read_text(encoding='utf-8'))
s4b = json.loads(Path('results/qwen4b/surrogate_metrics_4b.json').read_text(encoding='utf-8'))
c17 = json.loads(Path('results/qwen17b_causal/metrics_17b_causal.json').read_text(encoding='utf-8'))
ew2 = json.loads(Path('results/qwen4b/effect_weighted.json').read_text(encoding='utf-8'))

h80 = m17['cohort_tables']['heldout_80']['p99.9_plus']
b = s4b['benchmarks']
g1 = s4b['gap_decomposition_summary']['correlations']
b17 = c17['benchmarks_17b']
s17 = c17['structural_checks_17b']
q4 = ew2['qwen3_4b']
q17 = ew2['qwen3_17b']

print('\n=== Disk-Verified Headline Metrics ===')
print(f"Part 1 (Qwen3-1.7B heldout_80 p99.9+ terra): own={h80['own']['acc']['mean']:.3f}, token_only={h80['token_only']['acc']['mean']:.3f}, no_label={h80['no_label']['acc']['mean']:.3f}, wrong={h80['wrong']['acc']['mean']:.3f}")
print(f"Part 2 (Qwen3-4B n=278 Ulto85): top100 vs sim_cal_rec rho={b['b0_top100_terra']['sim_cal_rec']['spearman_rho']:.3f}, active rho={b['b0_top100_terra']['sim_cal_rec_active']['spearman_rho']:.3f}")
print(f"EWL (Qwen3-4B n={q4['n_kl_neurons']}): full={q4['all_kl_neurons']['ewl_full_energy_merge']['mean']:.3f}, 2-sided={q4['bipolar_only']['ewl_full_energy_merge']['mean']:.3f}, >=p75={q4['all_kl_neurons']['ewl_ge_p75_renorm_energy_merge']['mean']:.3f}, p99.9+ lift={q4['all_kl_neurons']['top_p999_lift']['mean']:.3f}")
print(f"EWL (Qwen3-1.7B n={q17['n_neurons']}): full={q17['all_50']['ewl_full_equal_merge']['mean']:.3f}, >=p75={q17['all_50']['ewl_ge_p75_renorm_equal_merge']['mean']:.3f}, over_no_label={q17['all_50']['lift_over_no_label_floor']['ewl_raw_full_equal_merge']['mean']:+.3f}")
print(f"Exploratory 4B: Energy(>=20%) vs strong20 rho={g1['energy_ge_20pct_max_vs_strong20_rec']['spearman_rho']:.3f}; Energy(>=20%) sim rho={b['b1_energy_ge_20pct_max']['sim_cal_rec']['spearman_rho']:.3f}; WDet_top100 partial={b['s2_wdet_clamped_top100_terra']['partial_above_energy_ge_20pct_sim_cal_rec']['partial_spearman_rho']:+.3f}; 1-P(YES|ord) active rho={b['s3_one_minus_pyes_ord_terra']['sim_cal_rec_active']['spearman_rho']:.3f}")
print(f"Exploratory 1.7B (n=50): EvalEnergy vs KLShare rho={s17['pooled_200_eval_energy_vs_kl_share']['spearman_rho']:.3f}; p99.9+ vs sim rho={b17['b0_p999_terra']['sim_cal_rec']['spearman_rho']:.3f}; Energy(>=20%) vs sim rho={b17['b1_energy_ge_20pct_max']['sim_cal_rec']['spearman_rho']:.3f}")
print(f"API spend (4B exploratory + 1.7B causal on disk): ${s4b['surrogate_spend']['combined_spend_usd']:.4f} + ${c17['causal_17b_spend']['combined_spend_usd']:.4f} = ${s4b['surrogate_spend']['combined_spend_usd'] + c17['causal_17b_spend']['combined_spend_usd']:.4f}")


Figure                                            Dims (px)  Size (KB)
------------------------------------------------------------------------
figures/main_detection_vs_kl_by_band.png        3850x1700        379.2
figures/main_band_accuracy.png                  2300x1350        211.8
figures/w_band_accuracy.png                     4620x1800        500.9
figures/context_ablation.png                    4320x1740        223.8
figures/qwen4b_main_plain.png                   2600x1550        338.0
figures/qwen4b_top100_two_panel.png             3400x1500        491.5
figures/qwen4b_detection_vs_causal.png          3400x1500        521.3
figures/qwen4b_detection_vs_strong20.png        2300x1450        311.2
figures/qwen4b_gap_decomposition.png            3600x1525        567.2
figures/qwen4b_surrogate_comparison.png         3700x1600        491.9
figures/cross_model_17b_vs_4b.png               3650x1600        360.7

=== Disk-Verified Headline Metrics ===
Part 1 (Qwen3-1.7B heldout_80 p99.9